In [51]:
import os
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime
import re

# -------------------------------
# CARGAR PERÍODOS DE NIEVE
# -------------------------------
def cargar_periodos_nieve(ruta):
    with open(ruta, 'r', encoding='utf-8') as f:
        contenido = f.read()
    torberas_periodos = {}
    bloques = contenido.split('✅ Procesado correctamente: ')[1:]
    for bloque in bloques:
        lineas = bloque.split('\n')
        nombre_raw = lineas[0].strip().replace('.csv', '')
        nombre = nombre_raw.replace('_periodo_nieve', '')
        periodos = []
        for linea in lineas:
            match = re.search(r'Snow in: (\d{2}/\d{2}/\d{4}) \| Snow off: (\d{2}/\d{2}/\d{4})', linea)
            if match:
                snow_in = datetime.strptime(match.group(1), '%d/%m/%Y')
                snow_off = datetime.strptime(match.group(2), '%d/%m/%Y')
                periodos.append((snow_in, snow_off))
        torberas_periodos[nombre] = periodos
    return torberas_periodos

# Ruta al txt
snow_txt_path = r"M:\material_isa\estadisticos\_Turberas\day_in_off_snow.txt"
periodos_nieve = cargar_periodos_nieve(snow_txt_path)

# Configuración
input_dir = r"M:\material_isa\estadisticos\_Turberas\subset_sinSNOW"
salida_dir = r"M:\material_isa\estadisticos\_Turberas\graficas_comparativas"
os.makedirs(salida_dir, exist_ok=True)

# Paleta de colores fijos
colores = {
    "Pleta_de_Molieres": "#8c564b", # marrón
    "Clots_de_Rialba": "#ff7f0e",  # naranja
    "Estanyeres": "#2ca02c",       # verde
    "Bassa_Nera": "#1f77b4",       # azul
    "Plan_deth_Horn": "#d62728",   # rojo
    "Pla_de_Sartari": "#9467bd",    # morado
    
}

# Formatos de fecha
def parse_fecha(fecha_str, nombre):
    if nombre == "Estanyeres":
        return datetime.strptime(fecha_str.strip(), "%d/%m/%Y")
    else:
        return datetime.strptime(fecha_str.strip(), "%Y-%m-%d")

# Función para graficar un índice para todas las turberas
def graficar_todas_las_turberas(indice, nombre_salida):
    plt.figure(figsize=(14, 6))
    for file in os.listdir(input_dir):
        if file.endswith("_sin_periodo_nieve.csv"):
            nombre = file.replace("_sin_periodo_nieve.csv", "")
            df = pd.read_csv(os.path.join(input_dir, file), sep=";", encoding="latin1")
            if indice not in df.columns:
                continue

            df["FECHA"] = df["FECHA"].astype(str).str.strip()
            df = df[df["FECHA"] != ""]
            try:
                df["FECHA_dt"] = df["FECHA"].apply(lambda x: parse_fecha(x, nombre))
            except Exception as e:
                print(f"❌ Error en fechas para {nombre}: {e}")
                continue
            df = df.dropna(subset=["FECHA_dt", indice])

            plt.scatter(df["FECHA_dt"], df[indice], s=8, label=nombre.replace("_", " "), color=colores.get(nombre, None), alpha=0.7)

    plt.title(f"{indice} – Todas las Turberas")
    plt.xlabel("Fecha")
    plt.ylabel(indice)
    plt.ylim(0, 1)
    plt.grid(True)
    plt.legend(title="Turbera", loc="best")
    
        # Dibujar recuadros grises de períodos de nieve (comunes a todas las turberas)
    for periodos in periodos_nieve.values():
        for snow_in, snow_off in periodos:
            plt.axvspan(snow_in, snow_off, color='gray', alpha=0.15)

    plt.text(1.01, 1.02, "Período SNOW", transform=plt.gca().transAxes,
             fontsize=9, ha='right', color='gray',
             bbox=dict(boxstyle="round,pad=0.2", facecolor='lightgray', edgecolor='none', alpha=0.5))

    plt.tight_layout()

    path_final = os.path.join(salida_dir, f"{nombre_salida}_v2.png")
    plt.savefig(path_final, dpi=150)
    plt.close()
    print(f"✅ Guardado: {path_final}")

# Ejecutar para los dos índices deseados
graficar_todas_las_turberas("NDVI", "NDVI_todas_turberas")
graficar_todas_las_turberas("NDWI_v", "NDWI_v_todas_turberas")




# --------------------- graficos lineas de regresion
import os
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime
from statsmodels.nonparametric.smoothers_lowess import lowess
from scipy.interpolate import UnivariateSpline
import numpy as np


# --- CONFIGURACIÓN ---
input_dir = r"M:\material_isa\estadisticos\_Turberas\subset_sinSNOW"
salida_dir = r"M:\material_isa\estadisticos\_Turberas\graficas_comparativas"

turberas = {
    "Bassa_Nera": "Bassa Nera",
    "Clots_de_Rialba": "Clots de Rialba",
    "Estanyeres": "Estanyeres",
    "Pla_de_Sartari": "Pla de Sartari",
    "Plan_deth_Horn": "Plan deth Horn",
    "Pleta_de_Molieres": "Pleta de Molieres"
}

colores = {
    "Bassa_Nera": "#1f77b4",
    "Clots_de_Rialba": "#ff7f0e",
    "Estanyeres": "#2ca02c",
    "Pla_de_Sartari": "#9467bd",
    "Plan_deth_Horn": "#d62728",
    "Pleta_de_Molieres": "#8c564b"
}

# --- FUNCIÓN PARA CARGAR Y PARSEAR CSV ---
def cargar_turbera(nombre, fecha_formato):
    archivo = os.path.join(input_dir, f"{nombre}_sin_periodo_nieve.csv")
    df = pd.read_csv(archivo, sep=";", encoding="latin1", decimal=",")
    df.columns = [c.strip().replace('\ufeff', '') for c in df.columns]

    df["FECHA"] = df["FECHA"].astype(str).str.strip()
    df = df[df["FECHA"] != ""]
    df["FECHA_dt"] = pd.to_datetime(df["FECHA"], format=fecha_formato, errors="coerce")
    return df.dropna(subset=["FECHA_dt"])

# --- FUNCIÓN PARA GRAFICAR CON SPLINE ---
from statsmodels.nonparametric.smoothers_lowess import lowess
from matplotlib.dates import date2num

def graficar_con_regresion(indice, nombre_archivo_out, ylim):
    plt.figure(figsize=(14, 6))
    for file in os.listdir(input_dir):
        if file.endswith("_sin_periodo_nieve.csv"):
            nombre = file.replace("_sin_periodo_nieve.csv", "")
            if nombre not in colores:
                continue

            df = pd.read_csv(os.path.join(input_dir, file), sep=';', encoding='latin1')
            if indice not in df.columns:
                continue

            df["FECHA"] = df["FECHA"].astype(str).str.strip()
            df = df[df["FECHA"] != ""]
            try:
                df["FECHA_dt"] = df["FECHA"].apply(lambda x: parse_fecha(x, nombre))
            except Exception as e:
                print(f"❌ Error en fechas para {nombre}: {e}")
                continue
            df = df.dropna(subset=["FECHA_dt", indice])

            x = df["FECHA_dt"]
            y = df[indice]
            color = colores[nombre]

            # Puntos originales
            plt.scatter(x, y, label=nombre.replace("_", " "), alpha=0.5, s=15, color=color)

            from matplotlib.dates import date2num, num2date

            # Convertir fechas a números
            x_num = date2num(x)
            y_array = np.array(y)

            # Ajustar spline (puedes ajustar 's' para controlar suavidad)
            spline = UnivariateSpline(x_num, y_array, s=0.006 * len(x))  # s es el factor de suavidad
            x_spline = np.linspace(x_num.min(), x_num.max(), 500)
            y_spline = spline(x_spline)

            # Dibujar spline suavizada
            plt.plot(num2date(x_spline), y_spline, '-', linewidth=1.5, color=color)



    plt.title(f"{indice} – Todas las Turberas (con SPLINE)")
    plt.xlabel("Fecha")
    plt.ylabel(indice)
    plt.ylim(*ylim)
    plt.grid(True)
    plt.legend(title="Turbera", loc="best")
    
        # Dibujar recuadros grises de períodos de nieve (comunes a todas las turberas)
    for periodos in periodos_nieve.values():
        for snow_in, snow_off in periodos:
            plt.axvspan(snow_in, snow_off, color='gray', alpha=0.15)

    plt.text(1.01, 1.02, "Período SNOW", transform=plt.gca().transAxes,
             fontsize=9, ha='right', color='gray',
             bbox=dict(boxstyle="round,pad=0.2", facecolor='lightgray', edgecolor='none', alpha=0.5))

    plt.tight_layout()

    nombre_out_v2 = nombre_archivo_out.replace(".png", "_v2.png")
    salida = os.path.join(salida_dir, nombre_out_v2)

    plt.savefig(salida, dpi=150)
    plt.close()
    print(f"✅ Guardado: {salida}")


# --- EJECUCIÓN PARA NDVI Y NDWI_v ---
graficar_con_regresion("NDVI", "NDVI_todas_turberas_regresion.png", (0.0, 1.0))
graficar_con_regresion("NDWI_v", "NDWI_v_todas_turberas_regresion.png", (0.0, 1.0))

print("\n🎯 Finalizado: gráficos con regresión LOWESS generados.")





✅ Guardado: M:\material_isa\estadisticos\_Turberas\graficas_comparativas\NDVI_todas_turberas_v2.png
✅ Guardado: M:\material_isa\estadisticos\_Turberas\graficas_comparativas\NDWI_v_todas_turberas_v2.png
✅ Guardado: M:\material_isa\estadisticos\_Turberas\graficas_comparativas\NDVI_todas_turberas_regresion_v2.png
✅ Guardado: M:\material_isa\estadisticos\_Turberas\graficas_comparativas\NDWI_v_todas_turberas_regresion_v2.png

🎯 Finalizado: gráficos con regresión LOWESS generados.
